In [54]:
from unike.module.model import RGCN, CompGCN
import sys

sys.path.extend(['.', '..'])

from q import link, drug_ent_indexs, indication_rel_index, mantle_cell_lymphoma_index, add_id

In [55]:
RGCN_model = RGCN(
	ent_tol = 121649,
	rel_tol = 22,
	dim = 200,
	num_layers = 2
)
RGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/RGCN_entrie_Accel_20250910-1000.pth")

In [56]:
CompGCN_model = CompGCN(
    ent_tol = 121649,
    rel_tol = 22,
    dim = 50
)
CompGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/CompGCN_entrie_Accel_20250910-1000.pth")

In [57]:
RGCN_result = add_id(link.link(
    drug_ent_indexs,
    [indication_rel_index],
    [mantle_cell_lymphoma_index],
    RGCN_model, 'cuda:0'
))

In [58]:
CompGCN_result = add_id(link.link(
    drug_ent_indexs,
    [indication_rel_index],
    [mantle_cell_lymphoma_index],
    CompGCN_model, 'cuda:0'
))

In [59]:
RGCN_result.to_csv("./RGCN.csv", index=False)
CompGCN_result.to_csv("./CompGCN.csv", index=False)

In [60]:
intersect_head = 100
RGCN_head_set = set(RGCN_result['head'].head(intersect_head).to_list())
CompGCN_head_set = set(CompGCN_result['head'].head(intersect_head).to_list())
intersect_set = RGCN_head_set & CompGCN_head_set
RGCN_result.query('head in @intersect_set')[['head', 'head_ent', 'head_id', 'uid']].to_csv(f"./intersect_result_{intersect_head}.csv", index=False)

## Add description

In [61]:
import pandas as pd
from tqdm.notebook import tqdm

In [62]:
descriptions = pd.read_csv('/home/wangtao/src/kg4rd/data/data_feature/drugbank.csv')

In [63]:

RGCN = RGCN_result
for idx, row in tqdm(RGCN.iterrows(), total=len(RGCN)):
    head_id = str(row['head_id']).split(':')[1]
    description = descriptions.query('id == @head_id')['description'].values[0]
    RGCN.loc[idx, 'description'] = description

RGCN.to_excel('RGCN_with_description.xlsx', index=False)


  0%|          | 0/9542 [00:00<?, ?it/s]

In [64]:
CompGCN = CompGCN_result
for idx, row in tqdm(CompGCN.iterrows(), total=len(CompGCN)):
    head_id = str(row['head_id']).split(':')[1]
    description = descriptions.query('id == @head_id')['description'].values[0]
    CompGCN.loc[idx, 'description'] = description

CompGCN.to_excel('CompGCN_with_description.xlsx', index=False)


  0%|          | 0/9542 [00:00<?, ?it/s]

In [65]:
# 合并为一个excel文件
intersect_df = RGCN_result.query('head in @intersect_set')[['head', 'head_ent', 'head_id', 'uid', 'description']]
with pd.ExcelWriter("./Result.xlsx", engine="openpyxl") as writer:
    intersect_df.to_excel(writer, sheet_name=f"intersect_head_{intersect_head}", index=False)
    RGCN.to_excel(writer, sheet_name="RGCN", index=False)
    CompGCN.to_excel(writer, sheet_name="CompGCN", index=False)